# Transformers

A transformer is a neural network architecture designed to model relationships between all tokens in a sequence **in parallel**, using a mechanism called **self-attention**. Instead of reading text one step at a time like RNNs/LSTMs, transformers let every token directly examine every other token in a single operation. Each token produces query, key, and value vectors that determine how much attention it should pay to other tokens; this allows the model to capture long-range dependencies, resolve ambiguity based on context, and form rich hierarchical representations of meaning. Because the entire sequence is processed simultaneously, transformers train quickly, scale extremely well, and form the backbone of modern NLP systems.

Transformers are built by stacking layers of **(1) multi-head self-attention**, **(2) feed-forward networks**, and **(3) normalization**. Depending on how masking is applied, the architecture can act as an encoder (bidirectional attention, used for understanding tasks like classification), a decoder (causal, used for generation like GPT), or an encoder–decoder (used for translation and summarization). More on this later. Training objectives such as masked language modeling (BERT) or autoregressive prediction (GPT) teach these layers to produce contextual embeddings that downstream models can use for tasks ranging from sentiment analysis to text generation.

## Attention Mechanism
In a NLP context, to determine the appropriate meaning of a word, a neural network should be able to **attend to** specific words from the rest of the sequence of words. To do this, an **attention mechanism** must weight factors whose values depend on specific input data. 

Given a set of input tokens $x_1, ..., x_N$ in some embedding space, we want to map it to another set $y_1, ..., y_n$ with the **same number of tokens**. Already we see a familiar concept of projection. In dimensionality reduction techniques like PCA, the goal is to linearly project data on one space (high-dimensional) to another space (low-dimensional). Here, we are doing more or less the same thing, except that we are keeping the dimensions the same (more on why in a second). With attention, the key intuition is that **dependence should be stronger for inputs $x_m$ that are important for deterining the modified representation of $y_n$**. We want
$$y_n=\sum^N_{m=1}a_{nm}x_m \quad \text{s.t.} \quad a_{nm}\geq 0, \sum^N_{m=1}a_{nm}=1$$
where $a_{nm}$ is the attention coefficient between the $n$ and $m$-th token/embedding. We can see that we are representing $y_n$ simply as a convex linear combination of $x_m$, $m=1,...,N$. Intuitively, we want to see how other tokens $x_m$ contribute to the semantic meaning of a target token $x_n$.

## Self-attention
The most basic building blocks of **attention** are the **key**, **value**, and **query** vectors. Consider a video streaming service like Netflix. In order to find a movie to watch, a user might search for specific attributes (e.g. genre, rating, actors, etc.). In turn, Netflix will look through their movie database and recommend movies that match the desired attributes. Here, we can think of the key as the attributes of a movie, the value the corresponding movie itself (or movie title), and a query as the user search. More formally,

- **query**: vector representing element of interest or context we obtain info about.
- **key**: The actual content or meaning of the word that is retrieved once a match is found..
- **value**: Projection of the input data

In the context of transformers, each input vector $x_n$ is **both the value and the key** -- thus the name **self-attention**. This basically means that when we search for a movie, we are just using the movie title itself. 

An important (and familiar) notion to understand is that of similarity. We want to be able to determine how similar a query and a key vector are (e.g. whether the movie title query by the user matches the movie title key in the database). A simple measure of similarity we can use is the dot product between the query and the key, $x_n^\top x_m$. Now, turning back to our attention coefficients, we can define them using softmax:
$$a_{nm} = \frac{\exp(x_n^\top x_m)}{\sum^N_{m'=1}\exp(x_n^\top x_{m'})}$$
It should be noted that the softmax has no probabilistic interpretation here. Instead, it is used in order to satisfy the constrains of the attention coefficients. We can write this in matrix notation as
$$ Y = \text{Softmax}[X^\top X]X $$
where the softmax operator is elementwise.

### Network parameters
Right now, we have to issues:
1. We don't have any adjustable parameters
2. All feature values within a token vecto plays an equal role in determining attention coefficients

We can address both issues by definiing modified feature vectors given by linear transformations. So, in place of the data matrices, we will have
- **query**: $Q=XW^{(q)}$, where $W^{(q)} \in \mathbb{R}^{D\times D}$
- **key**: $K=XW^{(k)}$, where $W^{(k)} \in \mathbb{R}^{D\times D}$
- **value**: $V=XW^{(v)}$, where $W^{(v)} \in \mathbb{R}^{D\times D}$. Note that the dimension of the value matrix doesn't have to be the same as key and query, in which case we can express the dimension as $D_v$.

Again, these are just simple linear transformations, but it turns out that they have a lot of expressive power. In particular, whereas the query weights $W^{(q)}$ encode how to express what a token might look for, the key weights $W^{(k)}$ encode the best way information might be offered, and the value weight $W^{(v)}$ encode the best way that information should be served. Without these projections, attention would be fixed and powerless; with them, it becomes expressive enough to learn syntax, semantics, and long-range dependencies.

It follows, then, that the updated expression for the self-attention output is
$$Y=\text{Softmax}[QK^\top]V$$
where $QK^\top \in \mathbb{R}^{N\times N}$ and $Y\in \mathbb{R}^{N\times D_v}$.

### Scaled self-attention
One final wrinkle in the attention block is the idea of scaling. The idea here is that gradients of softmax become exponentially small for inputs with high magnitude (think vanishing gradients). So, we have to rescale the product of the query and key vectors before applying softmax.

Assuming that elements of the query and key vectors are all independent random variables with mean 0 and unit variance, then $q^\top k = \sum^D_{i=1}q_ik_i$. It can be shown that the mean of $q^\top k$ is 0 and the variance is $D$. Thus, we normalize (or standardize) the argument to the softmax using the standard deviation:
$$Y = \text{Attention}(Q, K, V) = \text{Softmax}\left[\frac{QK^\top}{\sqrt{D}}\right]V$$

## Multi-headed attention
With the self-attention block above, we can capture a particular semantic pattern (e.g. adjectives before nouns). But what if there are multiple patterns of attention that are relevant at the same time? The main idea here is to use multiple attention blocks stacked on top of each other in parallel. This is known as **multi-headed attention**. Look [here](https://medium.com/data-science/transformers-explained-visually-part-3-multi-head-attention-deep-dive-1c1ff1024853) for more details.

Suppose we have $H$ attention heads where
$$H_h = \text{Attention}(Q_h, K_h, V_h), \; h=1,\dots,H$$
where each $H_h \in \mathbb{R}^{N \times d_k}$, $d_k=D/H$. More concretely, this means the query, key, and value matrices all have separate weight matrices:
$$Q_h = XW^{(q)}_h \quad\quad K_h = XW^{(k)}_h \quad\quad V_h = XW^{(v)}_h$$
We then stack all of the heads on top of each other, which creates a 2D tensor (matrix)
$$\text{Concat}(H_1, \dots, H_h) \in \mathbb{R}^{N \times (H * d_k)} = \mathbb{R}^{N \times D}$$

Notice the use of the terminology "tensor." This is because that even though a single attention head output is 2D, real models often have batches, which introduce a 3rd dimension. This would mean t working with tensors $\mathbb{R}^{\text{batch} \times N \times D}$.

The number of heads is a hyperparameter. Too many heads may over-parameterize or redundantly split information, too few heads may limit representation richness.

## Normalization

## Positional Encoding
So far in the transformer architecture, we have found a way to capture semantic meaning based on queries "attending to" keys. However, it is unclear how the attention block captures the relative ordering of the words. In other words, the attention mechanism tells us "what" the tokens mean, but not "where". As of right now, the output of the attention block is equivariant wrt input permutations. For example, "The food was good, not bad at all" and "The food was bad, not good at all" result in the same outputs. Thus, we need a way to encode token orderings, while also retaining the powerful properties of the attention mechanism.

One way to do this is as follows:
$$\tilde{x}_n = x_n + r_n$$
where $r_n$ is the **positional encoding**. Intuitively, the positional encoding simply represents the relative position of a token within a sequence. The reason we can perform the addition without corrupting the original embedding is because random uncorrelated vectors in high dimension tend to be close to orthogonal. So we can capture the semantic content and geometric location into one unified vector. To construct positional embeddings $\set{r_n}$, first note that an ideal encoding should provide:

1. Provide a unique embedding for each position.
2. Be bounded -- position doesn't dominate influence over attention.
3. Generalize to longer sequences.
4. Have a consistent way to express the number of steps between any two input vectors irrespective of the absolute position.

The original paper uses a technique based on sinusoidal functions. For the $n$-th position,
$$r_{ni} = 
\begin{cases} 
      \sin\left(\frac{n}{L^{i/D}}\right) & \text{if $i$ is even} \\
      \cos\left(\frac{n}{L^{(i-1)/D}}\right) & \text{if $i$ is odd}
   \end{cases}
$$
This representation has the property that elements of $r_n$ lie in $(-1, 1)$. It also has the property that learned positional embeddings can be defined up to the maximum training length, so we can evaluate $r_n$ for any $n \in \mathbb{N}$. Finally, the most important property is the way that $r_n$ expresses relative distances linearly. Because we can express $sin(a+b)$ and $cos(a+b)$ as a linear combination of $sin(a)$ and $cos(a)$, this means we can express $r_{n+k}$ as a linear transformation of $r_n$. This means that the attention layers can learn relative position relationships using matrix multiplication alone. In other words, The model doesn’t need to memorize absolute positions. It can reason about “how far apart are these two tokens?” using only dot products and linear maps.

## MLP layer

Once we have gone through the attention block, we also introduce a multi-layer perceptron (MLP) layer, which helps introduce non-linearity. For simplicity, the MLP layer just refers to a two-layer neural network (although it could be much deeper) with ReLU activation functions. So, the MLP layer takes the output of the attention block (which has the same dimension as the original input) and uses it as input for a feed-forward step.

### Information storage
I really like the interpretation of [3B1B's video](https://www.3blue1brown.com/lessons/mlp) on the topic, where he introduces the MLP as a component that stores facts. Suppose that a token encodes the phrase "Michael Jordan" (this isn't actually what's happening and is for intuition purposes only). When entering the MLP block, each vector that was originally associated with one of the tokens from the input text is going to go through a short series of operations that will result in another vector of the same dimension. These operations happen independently for each vector, meaning they don't exchange any information with each other (that's already accomplished with attention). Mathematically, this means that for attention outputs $y_1, \dots, y_N \in \mathbb{R}^D$, the neural network outputs:
$$ \text{MLP}(y_i) = W_2\sigma(W_1 y_i + b_1) + b_2$$ 
Let's look at each layer step in a little bit more detail. The first layer performs a linear expansion
$$h_i = W_1y_i + b_i$$
where $W_1$ lives in a higher dimensional space than the original data. This is essentially projecting the data to higher dimension to extract patterns that we don't have in the original space. In our example, this might mean that we project the concept of "Michael Jordan" into a higher-dimensional space where we can make associations (e.g. "basketball", "goat", etc.). We then introduce non-linearity through the ReLU activation
$$z_i = \sigma(h_i) = ReLU(h_i)$$
This essentially acts as a thresholding mechanism where the neuron only "activates" if the the input aligns sufficiently with some underlying pattern. So, the name "Michael Jordan" leads to an acitvation. The second layer acts as a "compression" layer, where recombine detected features, project back to the embedding space (dim D), producing new semantic features:
$$o_i = W_2 z_i + b_2$$
If there is a column (a direction) aligned with “basketball” and if that neuron is active, the down-projection will add a positive component in the “basketball” direction to the output vector. In this way, the MLP blocks provide “extra capacity to store facts.”

It is important to mention that the above example is very idealized in the sense that real models don't actually store facts. Instead, there is a phenomena of **superposition**, where real networks cram many “features/facts” into overlapping (non-orthogonal) directions, because the number of possible facts is astronomically larger than the embedding dimension. That means any single neuron is not a clean “Michael Jordan detector,” but rather mixes many latent micro-features — a “distributed representation.” So, real information storage is likely distributed, entangled, and uses superposition. The takeaway here is that MLPs are good at “factual recall” or “knowledge lookup” precisely because they can **act like** huge lookup tables embedded in weight matrices that store long-term knowledge structures.

Also recall that this is happening in parallel (think feed forward computation when vectors are independent) for all vectors $i=1,...,N$, which makes this process relatively efficient.

### Non-linearity
The above intuition can be a little bit confusing. To make things a little bit simpler, we can think of the MLP as just injecting non-linearity into model via the activation function, in true neural network fashion. The attention mechanism creates linear combos of the data. While there is some non-linearity introduced via the softmax function in the attention weights, we can enhance the flexibility of the transfomer by processing the output via the neural network.

## Transformer Layer

With all of the above model ingredients, we can officially form a transformer layer.

![transformer layer](transformer_layer.jpeg)

### Residual connections + MLP
The residual connections are direct connections from the original input ata directly to the output that bypasses the multi-headed attention block. They help preserve the original structure, which can get corrupted as representations flow from layer to layer.

$$ Z = \text{LayerNorm}[Y(X) + X]\quad\quad\text{or}\quad\quad Z=Y(\text{LayerNorm}[X]) + X$$

In each case, $Z$ has the same dimensionality as the input $X$. We can then send $Z$ through the MLP component

$$ \tilde{X} = \text{LayerNorm}[\text{MLP}(Z) + Z]\quad\quad\text{or}\quad\quad \tilde{X}=\text{MLP}(\text{LayerNorm}[Z]) + Z$$

Again, note that the $\text{MLP}$ operator is applied individually to each token, where the weights are the same across all. This is also known as **position-wise feed forward**:
$$ 
\begin{align}
    \text{MLP}(Z) &= \begin{bmatrix}
           \text{MLP}(Z_1) \\
           \vdots \\
           \text{MLP}(Z_N)
         \end{bmatrix}
\end{align}
$$
Because the MLP is still expressed as a single neural network, we can still use backprop for training the weights.

Each layer represents a pass through the attention, layer norm, and MLP components. The output of which captures both the semantic meaning of the words itself as well as the non-linear associations between those words and other concepts. We can iteratively refine both aspects by feeding the input through many transformer layers until we get to an output with extremely rich language context.

### Hyperparameter checklist

Some important parameters to consider when developing a Transfomer model:
- **Vocabulary Size ($|V|$)**: In Transformers, each input token is represented as an entry in a vocabulary V. Large vocabularies can cover more surface form variants of words, but may lead to increased storage requirements.
- **Embedding Size ($D$)**. Every token is represented as a $D$-dimensional real-valued vector. As presented above, this vector is the sum of the token embedding, positional embedding, and segment embedding, all of which are also $D$-dimensional real-valued vectors.
- **Hidden Size ($D_h$)**: The input and output of a sub-layer are of dimensions $D_h$. Most of the hidden states of a sub-layer are also $D_h$-dimensional vectors. In almost all modern transformers, the hidden size is equal to the embedding size.
- **Number of Heads ($n_{\text{head}}$)**: In self-attention sub-layers, one needs to specify the number of heads used in multi-head self-attention. The larger this number is, the more sub-spaces attention is performed. In practical systems, we often set $n_{\text{head}} \geq 4$.
- **FFN Hidden Size ($D_{\text{FFN}}$)**: The size of the hidden layer of the FFNs used in Transformers is
typically larger than . For example, a typical setting is $D_{\text{FFN}} = 4D$. For larger Transformers, such as recent large models, $D_{\text{FFN}}$ may be set to a very large value.
- **Model Depth ($L$)**: Using deep networks is an effective way to improve the expressive power of Transformers. For many models, $L$ is typically set to 12 or 24. However, networks with even greater depth are also feasible and can be applied for further enhancements.
